# HVAC Prediction: Simplified Workflow

This notebook provides a streamlined workflow for training, optimizing, and ensembling an LSTM model for HVAC power consumption prediction.

**Workflow:**
1.  **Setup:** Import libraries, define configuration, helper functions, dataset, and model architecture.
2.  **Data Preparation:** Load, preprocess, scale data, and create PyTorch DataLoaders.
3.  **Hyperparameter Optimization (Optuna):** Tune model hyperparameters to find the best combination based on validation performance.
4.  **Ensemble Training (Bagging):** Train multiple models using the best hyperparameters found, each with a different random seed.
5.  **Ensemble Prediction:** Use the trained ensemble to make predictions on the test set and generate a submission file.


## 1. Setup

Import necessary libraries, define configuration parameters, utility functions, the dataset class, and the model architecture.

In [19]:
# --- Imports ---
import os
import json
import random
import joblib
import pprint
import argparse # Used for Namespace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
from tqdm import tqdm # Use notebook-friendly tqdm

In [4]:
# --- Configuration ---
# Simplified configuration using a dictionary
config = {
    # Data/Paths
    "train_csv": "dataset/train_split.csv",
    "test_csv": "dataset/test_split.csv", # Used for evaluation during training/optimization
    "predict_features_csv": "dataset/test_features.csv", # Used for final prediction
    "output_dir": "models_lstm_simplified", # Base directory for saving models/scalers
    "results_dir": "results_lstm_simplified", # Base directory for saving results/submissions
    "base_feature_cols": [
        'temperature_exterieure', 'humidite', 'ensoleillement',
        'vitesse_vent', 'direction_vent', 'temperature_interieure'
    ],
    "target_col": 'puissance_cvac',

    # Model Defaults (can be overridden by Optuna)
    "seq_len": 16,
    "hidden_size": 128,
    "num_layers": 2,
    "dropout": 0.2,

    # Training Defaults (can be overridden by Optuna)
    "batch_size": 64,
    "num_epochs": 50, # Max epochs for a single trial
    "lr": 1e-3,
    "loss_function": "Huber", # 'MSE' or 'Huber'
    "val_split_ratio": 0.15,

    # Feature Engineering
    "use_time_features": True,

    # Training Enhancements
    "use_lr_scheduler": True,
    "lr_scheduler_patience": 5,
    "lr_scheduler_factor": 0.5,
    "use_early_stopping": True,
    "early_stopping_patience": 10,
    "use_gradient_clipping": True,
    "gradient_clip_max_norm": 1.0,

    # Optimization (Optuna)
    "n_trials": 30, # Number of Optuna trials
    "optuna_study_name": "hvac_optimization_lstm_simplified",
    "optuna_storage_url": None, # e.g., "sqlite:///hvac_optuna_lstm_simplified.db"

    # Ensemble (Bagging)
    "ensemble_seeds": [42, 123, 456, 789, 1011], # Seeds for ensemble members
    "ensemble_artifacts_file": "ensemble_artifacts.json", # File to store paths for ensemble models
    "final_submission_filename": "submission_ensemble.csv",
}

# Display config
pprint.pprint(config)

{'base_feature_cols': ['temperature_exterieure',
                       'humidite',
                       'ensoleillement',
                       'vitesse_vent',
                       'direction_vent',
                       'temperature_interieure'],
 'batch_size': 64,
 'dropout': 0.2,
 'early_stopping_patience': 10,
 'ensemble_artifacts_file': 'ensemble_artifacts.json',
 'ensemble_seeds': [42, 123, 456, 789, 1011],
 'final_submission_filename': 'submission_ensemble.csv',
 'gradient_clip_max_norm': 1.0,
 'hidden_size': 128,
 'loss_function': 'Huber',
 'lr': 0.001,
 'lr_scheduler_factor': 0.5,
 'lr_scheduler_patience': 5,
 'n_trials': 30,
 'num_epochs': 50,
 'num_layers': 2,
 'optuna_storage_url': None,
 'optuna_study_name': 'hvac_optimization_lstm_simplified',
 'output_dir': 'models_lstm_simplified',
 'predict_features_csv': 'dataset/test_features.csv',
 'results_dir': 'results_lstm_simplified',
 'seq_len': 16,
 'target_col': 'puissance_cvac',
 'test_csv': 'dataset/test_split.csv',

In [6]:
# --- Utility Functions ---
def add_time_features(df):
    """Adds time-based features to the DataFrame."""
    if 'date' not in df.columns:
        raise ValueError("'date' column is required but not found.")
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    return df

def set_seed(seed):
    """Sets random seed for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [7]:
# --- Core Components ---
class HVACDataset(Dataset):
    """PyTorch Dataset for HVAC sequences."""
    def __init__(self, sequences, targets):
        self.X = sequences.astype(np.float32)
        self.y = targets.astype(np.float32).reshape(-1, 1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    """LSTM Model for time series prediction."""
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super().__init__()
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=lstm_dropout)
        self.bn = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_time_step_out = out[:, -1, :]
        bn_out = self.bn(last_time_step_out)
        dropout_out = self.dropout(bn_out)
        final_out = self.fc(dropout_out)
        return final_out

def create_sequences(features, consumption, seq_len):
    """Creates input sequences and targets."""
    X, y = [], []
    N = len(consumption)
    # Target is consumption 16 steps after the *end* of the input sequence
    target_offset = 15
    for i in range(N - seq_len - target_offset):
        X.append(features[i:i+seq_len])
        y.append(consumption[i + seq_len + target_offset])
    return np.array(X), np.array(y)

## 2. Data Preparation

Load the training and testing data, apply scaling, generate time features (optional), create sequences, split into train/validation sets, and prepare DataLoaders.

In [8]:
def load_and_prepare_data(cfg):
    """Loads, preprocesses, and prepares data loaders."""
    print("Loading and preparing data...")
    
    # --- Load Data ---
    df_train_val = pd.read_csv(cfg['train_csv'], parse_dates=['date'])
    df_test = pd.read_csv(cfg['test_csv'], parse_dates=['date'])
    df_train_val = df_train_val.sort_values('date').reset_index(drop=True)
    df_test = df_test.sort_values('date').reset_index(drop=True)

    # --- Feature Engineering ---
    all_feature_cols = list(cfg['base_feature_cols']) # Copy
    if cfg['use_time_features']:
        df_train_val = add_time_features(df_train_val)
        df_test = add_time_features(df_test)
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols.extend(time_feature_cols)

    feat_train_val = df_train_val[all_feature_cols].values
    targ_train_val = df_train_val[cfg['target_col']].values
    feat_test = df_test[all_feature_cols].values
    targ_test = df_test[cfg['target_col']].values

    # --- Scaling ---
    feat_scaler = MinMaxScaler()
    feat_train_val_s = feat_scaler.fit_transform(feat_train_val)
    feat_test_s = feat_scaler.transform(feat_test)

    targ_scaler = MinMaxScaler()
    targ_train_val_s = targ_scaler.fit_transform(targ_train_val.reshape(-1,1)).flatten()
    targ_test_s = targ_scaler.transform(targ_test.reshape(-1,1)).flatten()

    # --- Create Sequences ---
    X_train_val, y_train_val = create_sequences(feat_train_val_s, targ_train_val_s, cfg['seq_len'])
    X_test, y_test = create_sequences(feat_test_s, targ_test_s, cfg['seq_len'])

    # --- Split Train/Validation ---
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=cfg['val_split_ratio'], shuffle=False
    )

    # --- DataLoaders ---
    train_ds = HVACDataset(X_train, y_train)
    val_ds = HVACDataset(X_val, y_val)
    test_ds = HVACDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True, pin_memory=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False, pin_memory=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False)

    print(f"Data loaded: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")
    print(f"Input feature size: {X_train.shape[-1]}")

    # Store input size in config for model building
    cfg['input_size'] = X_train.shape[-1]

    return train_loader, val_loader, test_loader, feat_scaler, targ_scaler

# Load the data
train_loader, val_loader, test_loader, feature_scaler, target_scaler = load_and_prepare_data(config)

Loading and preparing data...
Data loaded: Train=23800, Val=4201, Test=6977
Input feature size: 11


## 3. Hyperparameter Optimization (Optuna)

Define the training and evaluation logic for a single trial, then use Optuna to search for the best hyperparameters.

In [9]:
def run_training_trial(trial_cfg, train_loader, val_loader, test_loader, device, trial_num=None, random_seed=None):
    """
    Runs a single training and evaluation trial.
    Saves the best model, scalers, and results.
    Returns the validation MAE (for Optuna), test MAE, and paths to saved artifacts.
    """
    # --- Setup & Seeding ---
    current_seed = random_seed if random_seed is not None else random.randint(1, 10000)
    set_seed(current_seed)
    trial_cfg['current_seed'] = current_seed  # Store the seed used

    # --- File Naming & Directories ---
    output_dir = trial_cfg['output_dir']
    results_dir = trial_cfg['results_dir']
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    run_id = f"trial_{trial_num}_seed_{current_seed}" if trial_num is not None else f"seed_{current_seed}"

    model_filename = f"model_{run_id}.pth"
    feature_scaler_filename = f"feature_scaler_{run_id}.joblib"
    target_scaler_filename = f"target_scaler_{run_id}.joblib"
    results_filename = f"results_{run_id}.csv"

    model_path = os.path.join(output_dir, model_filename)
    feature_scaler_path = os.path.join(output_dir, feature_scaler_filename)
    target_scaler_path = os.path.join(output_dir, target_scaler_filename)
    results_path = os.path.join(results_dir, results_filename)

    # --- Build Model, Criterion, Optimizer, Scheduler ---
    model = LSTMModel(
        input_size=trial_cfg['input_size'],
        hidden_size=trial_cfg['hidden_size'],
        num_layers=trial_cfg['num_layers'],
        dropout=trial_cfg['dropout']
    ).to(device)

    criterion = nn.HuberLoss() if trial_cfg['loss_function'] == 'Huber' else nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=trial_cfg['lr'])

    scheduler = None
    if trial_cfg['use_lr_scheduler']:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=trial_cfg['lr_scheduler_factor'],
            patience=trial_cfg['lr_scheduler_patience'], verbose=False
        )

    # --- Training Loop ---
    best_val_loss = float('inf')
    epochs_no_improve = 0
    pbar_epochs = tqdm(range(1, trial_cfg['num_epochs'] + 1), desc=f'Run {run_id} Epochs', leave=False)

    for epoch in pbar_epochs:
        # Train
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            if trial_cfg['use_gradient_clipping']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=trial_cfg['gradient_clip_max_norm'])
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
        avg_train_loss = train_loss / len(train_loader.dataset)

        # Validate
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                out = model(Xb)
                loss = criterion(out, yb)
                val_loss += loss.item() * Xb.size(0)
        avg_val_loss = val_loss / len(val_loader.dataset)

        pbar_epochs.set_postfix({'TrL': f'{avg_train_loss:.5f}', 'VlL': f'{avg_val_loss:.5f}'})

        if trial_cfg['use_lr_scheduler'] and scheduler:
            scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if trial_cfg['use_early_stopping'] and epochs_no_improve >= trial_cfg['early_stopping_patience']:
                print(f"Early stopping triggered after epoch {epoch} for run {run_id}")
                break

    # --- Evaluation on Test Set ---
    test_mae = float('inf')
    if os.path.exists(model_path):  # Only evaluate if a model was saved
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        preds_scaled, trues_scaled = [], []
        with torch.no_grad():
            for Xb, yb in test_loader:
                Xb = Xb.to(device)
                out_scaled = model(Xb).cpu().numpy()
                preds_scaled.append(out_scaled)
                trues_scaled.append(yb.numpy())

        preds_scaled = np.concatenate(preds_scaled)
        trues_scaled = np.concatenate(trues_scaled)
        preds = target_scaler.inverse_transform(preds_scaled).flatten()
        trues = target_scaler.inverse_transform(trues_scaled).flatten()

        test_mse = mean_squared_error(trues, preds)
        test_mae = mean_absolute_error(trues, preds)

        results_df = pd.DataFrame({'y_true': trues, 'y_pred': preds})
        results_df.to_csv(results_path, index=False)

        # Save scalers associated with the best model
        joblib.dump(feature_scaler, feature_scaler_path)
        joblib.dump(target_scaler, target_scaler_path)
        print(f"Run {run_id}: Best Val Loss: {best_val_loss:.5f}, Test MAE: {test_mae:.5f}. Artifacts saved.")

        # Prepare artifact info dictionary
        artifacts = {
            "model_path": model_path,
            "feature_scaler_path": feature_scaler_path,
            "target_scaler_path": target_scaler_path,
            "results_path": results_path,
            "val_loss": best_val_loss,
            "test_mae": test_mae,
            "params": trial_cfg
        }
        return best_val_loss, test_mae, artifacts
    else:
        print(f"Run {run_id}: No model saved (training might have failed or not improved).")
        return float('inf'), float('inf'), None

In [20]:
# --- Optuna Objective Function ---
def objective(trial, base_cfg, train_loader, val_loader, test_loader, device):
    """Optuna objective: suggest hyperparameters, run trial, return validation metric."""
    # Create a copy of the base config for this trial
    trial_cfg = base_cfg.copy()

    # Suggest hyperparameters
    trial_cfg['lr'] = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    trial_cfg['num_layers'] = trial.suggest_int("num_layers", 1, 4)
    trial_cfg['batch_size'] = trial.suggest_categorical("batch_size", [32, 64, 128])
    trial_cfg['hidden_size'] = trial.suggest_int("hidden_size", 32, 128, step=16)
    trial_cfg['dropout'] = trial.suggest_float("dropout", 0.0, 0.5)
    # Optional: trial_cfg['seq_len'] = trial.suggest_categorical("seq_len", [8, 16, 24])

    # Run the training trial
    val_metric, test_mae, artifacts = run_training_trial(
        trial_cfg, train_loader, val_loader, test_loader, device, trial_num=trial.number
    )

    # Store artifacts info in the trial for later retrieval
    if artifacts:
        trial.set_user_attr("artifacts", artifacts)
        trial.set_user_attr("test_mae", test_mae)  # Store test MAE for reference

    # Optuna minimizes the returned metric (validation loss)
    return val_metric

In [21]:
# --- Run Optuna Study ---
print("\n--- Starting Hyperparameter Optimization ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

study = optuna.create_study(
    study_name=config['optuna_study_name'],
    storage=config['optuna_storage_url'],
    direction="minimize",
    load_if_exists=True  # Resume study if storage is configured and study exists
)

# Pass fixed arguments (loaders, device) using a lambda
study.optimize(
    lambda trial: objective(trial, config, train_loader, val_loader, test_loader, device),
    n_trials=config['n_trials']
)

# --- Get Best Trial Results ---
best_trial = study.best_trial
best_artifacts = None
if best_trial:
    print(f"\nOptimization Finished.")
    print(f"Best Trial Number: {best_trial.number}")
    print(f"Best Validation Metric (Loss): {best_trial.value:.5f}")
    print(f"Best Hyperparameters:")
    pprint.pprint(best_trial.params)

    if "artifacts" in best_trial.user_attrs:
        best_artifacts = best_trial.user_attrs["artifacts"]
        print(f"Best Trial Test MAE: {best_artifacts['test_mae']:.5f}")
        print(f"Model saved to: {best_artifacts['model_path']}")
    else:
        print("Warning: Artifacts not found for the best trial.")
else:
    print("Optimization did not complete successfully or found no best trial.")

# Store best hyperparameters in the main config for the next step
if best_artifacts:
    config.update(best_trial.params)  # Update config with best hyperparams
    config['best_trial_artifacts'] = best_artifacts  # Store artifact info
    print("\nUpdated config with best hyperparameters found.")

[I 2025-04-19 01:55:36,023] A new study created in memory with name: hvac_optimization_lstm_simplified



--- Starting Hyperparameter Optimization ---
Using device: cpu
















































































































[W 2025-04-19 01:56:53,986] Trial 0 failed with parameters: {'lr': 0.0004110966929222281, 'num_layers': 3, 'batch_size': 32, 'hidden_size': 96, 'dropout': 0.20870310388311725} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/ben/Documents/Projects/ift3395-project/.venv/lib/python3.13/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/h8/fhnz3l2j61j0pj5d_0_zzfpr0000gn/T/ipykernel_55437/1675890804.py", line 15, in <lambda>
    lambda trial: objective(trial, config, train_loader, val_loader, test_loader, device),
                  ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/h8/fhnz3l2j61j0pj5d_0_zzfpr0000gn/T/ipykernel_55437/2972314180.py", line 16, in objective
    val_metric, test_mae, artifacts = run_trainin

KeyboardInterrupt: 

## 4. Ensemble Training (Bagging)

Train multiple models using the best hyperparameters identified by Optuna, each initialized with a different random seed. Store the artifact paths for each model.

In [ ]:
def run_ensemble_training(base_cfg, seeds, train_loader, val_loader, test_loader, device):
    """Trains multiple models with the same config but different seeds."""
    print(f"\n--- Starting Ensemble Training for Seeds: {seeds} ---")
    ensemble_artifacts = {}

    # Ensure best params are used if available, otherwise use defaults
    if 'best_trial_artifacts' in base_cfg:
        print("Using hyperparameters from the best optimization trial.")
        # Create a config for ensemble runs based on best params
        ensemble_cfg = base_cfg['best_trial_artifacts']['params'].copy()
        # Ensure essential paths/settings from base_cfg are kept
        ensemble_cfg.update({k: v for k, v in base_cfg.items() if k not in ensemble_cfg})
    else:
        print("Warning: Best trial artifacts not found. Using default config for ensemble.")
        ensemble_cfg = base_cfg.copy()

    for i, seed in enumerate(seeds):
        print(f"\nTraining ensemble member {i+1}/{len(seeds)} with seed {seed}...")
        # Ensure each run has a unique seed but uses the same hyperparameters
        member_cfg = ensemble_cfg.copy()
        
        # Run training trial for this seed
        val_metric, test_mae, artifacts = run_training_trial(
            member_cfg, train_loader, val_loader, test_loader, device, trial_num=f"ensemble_{i}", random_seed=seed
        )
        
        if artifacts:
            ensemble_artifacts[f"seed_{seed}"] = artifacts
            print(f"Ensemble member {i+1} (Seed {seed}) finished. Test MAE: {test_mae:.5f}")
        else:
            print(f"Ensemble member {i+1} (Seed {seed}) failed to train successfully.")
            
    # Save the collected artifact paths
    output_dir = base_cfg['output_dir']
    os.makedirs(output_dir, exist_ok=True)
    artifacts_path = os.path.join(output_dir, base_cfg['ensemble_artifacts_file'])
    with open(artifacts_path, 'w') as f:
        # Use a custom serializer for Namespace if needed, or convert params to dict
        def serialize_artifacts(artifacts_dict):
            serializable = {}
            for key, data in artifacts_dict.items():
                if isinstance(data['params'], argparse.Namespace):
                    data['params'] = vars(data['params'])
                serializable[key] = data
            return serializable
        json.dump(serialize_artifacts(ensemble_artifacts), f, indent=4)
        
    print(f"\nEnsemble training finished. Artifact details saved to: {artifacts_path}")
    return ensemble_artifacts

# Run Ensemble Training
ensemble_results = run_ensemble_training(
    config, 
    config['ensemble_seeds'], 
    train_loader, 
    val_loader, 
    test_loader, 
    device
)

## 5. Ensemble Prediction

Load the models trained during the ensemble phase, generate predictions on the final test dataset (`test_features.csv`), average the predictions, and save the submission file.

In [ ]:
def prepare_prediction_data(cfg, feature_scaler_path):
    """Loads and prepares the final test data for prediction."""
    print(f"\nLoading prediction features from: {cfg['predict_features_csv']}")
    df_pred = pd.read_csv(cfg['predict_features_csv'], parse_dates=['date'])
    ids = df_pred['ID'].values

    # --- Feature Engineering ---
    all_feature_cols = list(cfg['base_feature_cols'])
    if cfg['use_time_features']:
        df_pred = add_time_features(df_pred)
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols.extend(time_feature_cols)

    # Ensure columns match training
    features = df_pred[all_feature_cols].values 

    # --- Load Scaler and Scale ---
    try:
        feat_scaler = joblib.load(feature_scaler_path)
    except FileNotFoundError:
        print(f"Error: Feature scaler not found at {feature_scaler_path}")
        return None, None
    features_scaled = feat_scaler.transform(features)

    # --- Create Sequences (Simplified for Prediction) ---
    # For prediction, we often just need the latest features to predict the future point.
    # Here, we create sequences by tiling the last available feature vector.
    num_samples, num_features = features_scaled.shape
    seq_len = cfg['seq_len']
    X_pred_seq = np.zeros((num_samples, seq_len, num_features), dtype=np.float32)
    for i in range(num_samples):
        X_pred_seq[i, :, :] = np.tile(features_scaled[i, :], (seq_len, 1))
        
    print(f"Prediction data prepared: {X_pred_seq.shape[0]} samples.")
    return X_pred_seq, ids

In [ ]:
def run_ensemble_prediction(pred_cfg, ensemble_artifacts, device):
    """Generates predictions using the ensemble and averages them."""
    print(f"\n--- Running Ensemble Prediction ---")
    if not ensemble_artifacts:
        print("Error: No ensemble artifacts provided. Cannot predict.")
        return None

    all_predictions = []
    final_ids = None
    target_scaler = None  # Load the first target scaler

    # Load the first member's artifacts to prepare data
    first_member_id = next(iter(ensemble_artifacts))
    first_member_data = ensemble_artifacts[first_member_id]
    feature_scaler_path = first_member_data['feature_scaler_path']
    target_scaler_path = first_member_data['target_scaler_path']
    member_params = first_member_data['params']  # Use params from a member

    # Merge member params with base prediction config if needed (e.g., for seq_len)
    effective_cfg = pred_cfg.copy()
    effective_cfg.update(member_params)

    # Prepare prediction data using the first member's feature scaler
    X_pred_seq, final_ids = prepare_prediction_data(effective_cfg, feature_scaler_path)
    if X_pred_seq is None:
        return None
        
    # Load the target scaler (assume they are compatible across ensemble)
    try:
        target_scaler = joblib.load(target_scaler_path)
    except FileNotFoundError:
        print(f"Error: Target scaler not found at {target_scaler_path}")
        return None

    # Iterate through each ensemble member for prediction
    for member_id, member_data in tqdm(ensemble_artifacts.items(), desc="Ensemble Members"):
        model_path = member_data['model_path']
        member_params = member_data['params']  # Config used to train this member
        
        try:
            # Build model architecture based on this member's parameters
            model = LSTMModel(
                input_size=member_params['input_size'],
                hidden_size=member_params['hidden_size'],
                num_layers=member_params['num_layers'],
                dropout=member_params['dropout']
            ).to(device)
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval()

            # Predict using batches
            preds_scaled_list = []
            pred_dataset = TensorDataset(torch.from_numpy(X_pred_seq).float())
            pred_loader = DataLoader(pred_dataset, batch_size=pred_cfg.get('batch_size', 128), shuffle=False)

            with torch.no_grad():
                for Xb, in pred_loader:
                    Xb = Xb.to(device)
                    batch_preds_scaled = model(Xb).cpu().numpy()
                    preds_scaled_list.append(batch_preds_scaled)
            
            preds_scaled = np.concatenate(preds_scaled_list)
            preds = target_scaler.inverse_transform(preds_scaled).flatten()
            all_predictions.append(preds)

        except Exception as e:
            print(f"Error processing member {member_id}: {e}")

    # Average predictions
    if not all_predictions:
        print("Error: No predictions generated from any ensemble member.")
        return None
        
    print(f"\nAveraging predictions from {len(all_predictions)} members.")
    final_preds = np.mean(np.vstack(all_predictions), axis=0)

    # Save final submission
    results_dir = pred_cfg['results_dir']
    os.makedirs(results_dir, exist_ok=True)
    submission_path = os.path.join(results_dir, pred_cfg['final_submission_filename'])
    df_out = pd.DataFrame({'ID': final_ids, 'puissance_cvac_future': final_preds})
    df_out.to_csv(submission_path, index=False)
    print(f"Ensemble prediction finished. Submission file saved to: {submission_path}")
    return submission_path

# Run Ensemble Prediction
if ensemble_results:  # Check if ensemble training produced results
    submission_file_path = run_ensemble_prediction(config, ensemble_results, device)
else:
    print("Skipping ensemble prediction as ensemble training did not produce results.")